# Colab GPU Bridge: Ollama + ngrok static tunnel

Runs Ollama on Colab's free T4 GPU and exposes it over a **fixed** HTTPS URL, so your phone/VS Code client never needs reconfiguring after a Colab restart.

## Before you run this
1. **Runtime > Change runtime type > T4 GPU**, then Connect.
2. Create a free ngrok account at https://dashboard.ngrok.com/signup.
3. Grab your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken
4. In this notebook, click the key icon (Secrets) in the left sidebar and add a secret named `NGROK_AUTHTOKEN` with that value. Enable notebook access for it.
5. Reserve one free static domain: https://dashboard.ngrok.com/domains -> Create Domain. Copy the `*.ngrok-free.app` hostname into the CONFIG cell below.
6. Run all cells top to bottom (Runtime > Run all). Whenever Colab disconnects/resets, just Run all again -- the URL stays the same.

In [ ]:
# CONFIG -- edit these two lines
NGROK_DOMAIN = "your-reserved-domain.ngrok-free.app"  # from https://dashboard.ngrok.com/domains
MODEL = "llama3.1:8b"  # 8B fits the T4's 16GB VRAM comfortably; try "qwen2.5:14b-instruct-q4_K_M" for a bigger quantized model

!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Start the Ollama server in the background, then pull the model
import subprocess, time, requests

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/content/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)

for _ in range(30):
    try:
        requests.get("http://127.0.0.1:11434")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    raise RuntimeError("Ollama server did not come up -- check /content/ollama.log")

print("Ollama is up, pulling model (this can take a few minutes)...")
!ollama pull {MODEL}

In [ ]:
# Install ngrok, authenticate from the Secrets panel, and open the static tunnel
from google.colab import userdata

NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")

!curl -sSL https://ngrok-agent.s3.amazonaws.com/ngrok.asc | tee /etc/apt/trusted.gpg.d/ngrok.asc >/dev/null
!echo "deb https://ngrok-agent.s3.amazonaws.com buster main" | tee /etc/apt/sources.list.d/ngrok.list
!apt update -qq && apt install -y ngrok -qq

!ngrok config add-authtoken {NGROK_AUTHTOKEN}

import subprocess, time, requests

ngrok_proc = subprocess.Popen(
    ["ngrok", "http", "11434", "--domain", NGROK_DOMAIN, "--log", "stdout"],
    stdout=open("/content/ngrok.log", "w"),
    stderr=subprocess.STDOUT,
)

time.sleep(5)
url = f"https://{NGROK_DOMAIN}"
try:
    r = requests.get(url, timeout=10)
    print(f"Bridge is live: {url}  (status {r.status_code})")
except requests.exceptions.RequestException as e:
    print(f"Tunnel not responding yet, check /content/ngrok.log -- {e}")

print(f"\nPoint your client at: {url}")

## Optional: keep the session alive a little longer

Colab still disconnects idle free-tier sessions on its own schedule regardless of this cell -- this only stops *your* notebook from looking idle due to zero cell activity. Keep the interval modest; don't turn this into a tight polling loop, that's against Colab's free-tier usage terms.

In [ ]:
import time, datetime

try:
    while True:
        print(f"[{datetime.datetime.now().isoformat(timespec='seconds')}] bridge alive at https://{NGROK_DOMAIN}")
        time.sleep(300)
except KeyboardInterrupt:
    print("Stopped heartbeat -- Ollama and ngrok are still running in the background.")